# Đánh giá mô tả bằng mô hình chấm

Đánh giá phát biểu của mô tả theo kế hoạch chấm đã quy định, tách khỏi chấm độ chính xác câu trả lời.

In [ ]:
import os
import sys
from pathlib import Path

source_root = Path(os.environ.get("MEDVQA_SOURCE_ROOT", ".")).resolve()
if not (source_root / "medvqa").is_dir():
    raise FileNotFoundError("Set MEDVQA_SOURCE_ROOT to the extracted source directory.")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

## Cấu hình thí nghiệm

Chọn mô hình chấm, thư mục mô tả và bộ tham chiếu tương ứng. Dùng `PILOT` để kiểm tra định dạng đầu ra trước khi chạy `TEST`.

In [ ]:
from pathlib import Path

MODE = "TEST"
RUN_ID = "test_judge_J1"
JUDGE = "J1"
TASKS = ["J-AX1"]
OBSERVERS = None
PARTIAL_OBSERVERS = []
PILOT_MAX_UNITS_PER_TASK = 20
RQ2 = "AX1_SLAKE"
PLATFORM = None
DESIGN = "MAIN"
INPUT_ROOT = None
LOCK_FILE = None
RUN_PARAMS_FILE = None
OBS_DIRS = []
ANS_DIRS = []
GOLD_RAD_FILE = None
PRIOR_OUTPUT_DIRS = []
OUT_ROOT = None
SHARD_TIME_LIMIT_S = None
SHARD_INDEX, NUM_SHARDS = (0, 1)
MAX_ITEMS_THIS_SESSION = None
PRECISION = "PREC-0"
JUDGE_ENV = "ENV-1"
BACKEND = "hf"
INSTALL = False
MODULE_DIR = source_root

## Thực thi và lưu kết quả

In [ ]:
import json
import sys
import time
from collections import Counter, defaultdict

sys.path.insert(0, str(MODULE_DIR))

In [ ]:
from medvqa import common as U
from medvqa import execution as RUN

PLATFORM = (
    PLATFORM or RUN.MODEL_PLATFORM[{"J1": "J1", "J2": "J2", "J-ALT": "PHI"}[JUDGE]]
)
PLATFORM_CFG = RUN.platform_setup(PLATFORM)
INPUT_ROOT = Path(INPUT_ROOT or PLATFORM_CFG["input_root"])
OUT_ROOT = Path(OUT_ROOT or PLATFORM_CFG["out_root"])
SHARD_TIME_LIMIT_S = SHARD_TIME_LIMIT_S or PLATFORM_CFG["time_limit_s"]
LOCK_FILE = Path(LOCK_FILE or INPUT_ROOT / "." / "locked_inputs.sha256")

In [ ]:
from medvqa import conditions as CL

OBSERVERS = OBSERVERS or CL.observers_for(DESIGN)
FB = DESIGN in ("FB", "MAIN")
if DESIGN == "MAIN" and set(TASKS) - {"J-AX1"}:
    raise ValueError("MAIN: only J-AX1 (+ J-PROBE) is in scope for RQ2")
if DESIGN == "MAIN" and RQ2 != "AX1_SLAKE":
    raise ValueError(
        (
            "MAIN with RQ2='NONE': the judge is not run (RQ2 "
            "= future work, experiment configuration MAIN)"
        )
    )
RUN_PARAMS_FILE = Path(RUN_PARAMS_FILE or INPUT_ROOT / "." / "run_params.json")
RUN.verify_run_params_first(LOCK_FILE, RUN_PARAMS_FILE, MODE)
platform_report = RUN.check_platform(
    {"J1": "J1", "J2": "J2", "J-ALT": "PHI"}[JUDGE], PLATFORM, MODE, RUN_PARAMS_FILE
)
RUN.check_design(DESIGN, MODE, RUN_PARAMS_FILE)
ENV_PINS = RUN.env_pins_for(MODE, RUN_PARAMS_FILE, JUDGE_ENV)
if INSTALL and BACKEND == "hf":
    RUN.install_env(JUDGE_ENV, ENV_PINS)
env_report = RUN.verify_env(JUDGE_ENV, ENV_PINS, MODE)
print("environment:", env_report)

In [ ]:
from medvqa import conditions as CL
from medvqa import judge_scope as AX1
from medvqa import judges as JG
from medvqa import models as B
from medvqa import prompts as P
from medvqa import routing as R

PREP = INPUT_ROOT / "prep_out"
PROMPTS = INPUT_ROOT / "resources/protocol" / "prompts"
U.assert_no_sealed_inputs(
    [INPUT_ROOT] + list(OBS_DIRS) + list(ANS_DIRS) + list(PRIOR_OUTPUT_DIRS)
)
JKEY = {"J1": "J1", "J2": "J2", "J-ALT": "PHI"}[JUDGE]
ARMS = ["ARM-GEN", "ARM-ANA", "ARM-GRD"]
if MODE == "PILOT" and "J-OPEN" in TASKS:
    raise ValueError("J-OPEN needs test gold: not allowed in PILOT")

In [ ]:
COHORTS = RUN.cohorts_for(MODE)
SUFFIX = "test" if MODE == "TEST" else "dev"
ann_rel, probe_rel = (
    f"judge/annotations_slake_{SUFFIX}.json",
    "judge/jprobe_set.json" if MODE == "TEST" else "judge/jprobe_set_dev.json",
)
lock_files = RUN.prompt_lock_files(
    INPUT_ROOT, ["templates/judge_prompts.md", "field_trace.json"]
)
lock_files.update(
    RUN.prep_lock_files(
        INPUT_ROOT,
        [
            "route_metadata.json",
            "route_pairs.json",
            "panel_rq2.json",
            ann_rel,
            probe_rel,
        ]
        + [f"cohorts/{c}.jsonl" for c in COHORTS],
    )
)
if MODE == "TEST":
    lock_files["run_params.json"] = RUN_PARAMS_FILE
lock_report = U.verify_locked_inputs(LOCK_FILE, lock_files, MODE)
ax1_scope = ax1_plan = None
ax1_scope_sha = ax1_plan_sha = None
if MODE == "TEST":
    rp = U.read_json(RUN_PARAMS_FILE)
    if RUN.rp_get(rp, "precision", JKEY) != PRECISION:
        raise U.LockError("judge PRECISION differs from locked run_params.json")
    if JUDGE == "J-ALT" and (not rp.get("judge_fallback")):
        raise U.LockError("J-ALT used but run_params.judge_fallback is not set")
    scope_path = INPUT_ROOT / "artifacts" / "ax1_scope.json"
    if rp.get("ax1_scope") is not None or scope_path.is_file():
        if rp.get("ax1_scope") != {
            "version": AX1.SCOPE_VERSION,
            "configuration_ref": "REFERENCE_PANEL",
        }:
            raise U.LockError(
                (
                    "REFERENCE_PANEL scope requires the matching "
                    "locked run_params.ax1_scope declaration"
                )
            )
        if (
            TASKS != ["J-AX1"]
            or JUDGE not in ("J1", "J2")
            or PARTIAL_OBSERVERS
            or (set(OBSERVERS) != {"OBS-L", "OBS-Q", "OBS-G"})
        ):
            raise U.LockError(
                "REFERENCE_PANEL requires J1/J2, J-AX1 and all three MAIN observers"
            )
        plan_path = INPUT_ROOT / "artifacts" / f"ax1_plan_{JUDGE}.json"
        lock_files.update(
            {
                "artifacts/ax1_scope.json": scope_path,
                f"artifacts/ax1_plan_{JUDGE}.json": plan_path,
            }
        )
        lock_report = U.verify_locked_inputs(LOCK_FILE, lock_files, MODE)
        ax1_scope, ax1_plan = (U.read_json(scope_path), U.read_json(plan_path))
        ax1_scope_sha, ax1_plan_sha = (
            U.sha256_file(scope_path),
            U.sha256_file(plan_path),
        )
rows = RUN.load_cohort_rows(PREP, COHORTS)
q_by_img = defaultdict(list)
for r in rows:
    q_by_img[r["neutral_id"]].append(r)
ann = U.read_json(PREP / ann_rel)
probes = U.read_json(PREP / probe_rel)["probes"]
if MODE == "TEST":
    panel = U.read_json(PREP / "panel_rq2.json")
    panel_ids = [n for n in panel["neutral_ids"] if n in q_by_img]
    slake_ids = [n for n in panel["strata"]["SLAKE"] if n in q_by_img]
    required_ax1_ids = panel["strata"]["SLAKE"]
else:
    panel_ids = sorted(q_by_img)
    slake_ids = [
        n
        for n in panel_ids
        if any((r.get("strata", {}).get("dataset") == "SLAKE" for r in q_by_img[n]))
    ]
    required_ax1_ids = slake_ids
    probes = [p for p in probes if p["neutral_id"] in set(slake_ids)]
annotation_preflight = None
if "J-AX1" in TASKS:
    if ax1_scope is not None:
        AX1.validate_scope(
            ax1_scope,
            panel,
            ann,
            U.sha256_file(PREP / "panel_rq2.json"),
            U.sha256_file(PREP / ann_rel),
            U.sha256_file(PREP / probe_rel),
            probes,
            q_by_img,
        )
        eligible = set(ax1_scope["eligible_ids"])
        slake_ids = [n for n in slake_ids if n in eligible]
        required_ax1_ids = slake_ids
        AX1.validate_plan(
            ax1_plan,
            ax1_scope_sha,
            U.sha256_file(PREP / probe_rel),
            JUDGE,
            eligible,
            {p["probe_id"]: p["neutral_id"] for p in probes},
        )
    annotation_preflight = JG.validate_ax1_annotations(
        ann,
        set(required_ax1_ids) | {p["neutral_id"] for p in probes},
        MODE,
        cohort_ids=q_by_img,
    )
    print("J-AX1 annotation preflight:", annotation_preflight)
rm = U.read_json(PREP / "route_metadata.json")["records"]
rp_table = U.read_json(PREP / "route_pairs.json")
routing = {n: R.route_for(n, rm[n], rp_table) for n in panel_ids}
field_trace = json.loads((PROMPTS / "field_trace.json").read_text(encoding="utf-8"))
bank = P.ObserverPromptBank(PROMPTS)
JP = JG.JudgePrompts(PROMPTS / "templates" / "judge_prompts.md")
obs = RUN.load_records(OBS_DIRS, "OBS-v1") if OBS_DIRS else {}
print(
    MODE,
    "| panel images:",
    len(panel_ids),
    "| SLAKE (J-AX1):",
    len(slake_ids),
    "| probes:",
    len(probes),
    "| observations:",
    len(obs),
)

In [ ]:
units, skipped_missing = ([], Counter())


def get_obs(o, a, n):
    r = obs.get(f"OBS|{o}|{a}|{n}")
    if r is None:
        if o in PARTIAL_OBSERVERS:
            skipped_missing[o] += 1
            return None
        raise U.IntegrityError(f"missing observation {o} {a} {n}")
    return r


if "J-AX1" in TASKS:
    units += JG.build_ax1_units(slake_ids, probes, JUDGE, OBSERVERS, ARMS, get_obs)
if "J-AX2" in TASKS:
    for n in panel_ids:
        for o in OBSERVERS:
            for a in ARMS:
                r = get_obs(o, a, n)
                if r is not None:
                    units.append(
                        {
                            "key": f"J|{JUDGE}|J-AX2|{r['key']}|main",
                            "task": "J-AX2",
                            "nid": n,
                            "obs_key": r["key"],
                            "text": r["text"],
                            "obs_status": r["status"],
                        }
                    )
if "J-AX3" in TASKS:
    for n in panel_ids:
        for o in OBSERVERS:
            r = get_obs(o, "ARM-GRD", n)
            if r is None:
                continue
            for q in q_by_img[n]:
                units.append(
                    {
                        "key": f"J|{JUDGE}|J-AX3|{r['key']}#{q['qid']}|main",
                        "task": "J-AX3",
                        "nid": n,
                        "obs_key": r["key"],
                        "qid": q["qid"],
                        "question": q["question"],
                        "text": r["text"],
                        "obs_status": r["status"],
                    }
                )

In [ ]:
def lead_text(t):
    """Clean an open answer with the fixed first-line parsing rules."""
    for ln in str(t or "").splitlines():
        if ln.strip():
            return ln.strip().lstrip("\"'`*([{ ").rstrip(".").strip()
    return ""


if "J-OPEN" in TASKS:
    ans = RUN.load_records(ANS_DIRS, "ANS-v1")
    rad = [r for r in rows if r["cohort"] == "COH-RAD-TEST"]
    gate_missing = [
        (c["cell"], r["qid"])
        for c in CL.all_cells(design=DESIGN)
        for r in rad
        for o in (r["orders"] if r["answer_kind"] == "mcq" else ["-"])
        if f"ANS|{c['cell']}|{r['qid']}|{o}" not in ans
    ]
    if gate_missing:
        raise U.IntegrityError(
            (
                "J-OPEN gate: "
                f"{len(gate_missing)}"
                " required COH-RAD-TEST answers missing (e.g. "
                f"{gate_missing[:2]}"
                ")"
            )
        )
    with U.sealed_guard(allow={"gold_RAD_TEST.jsonl"}):
        gold = {g["qid"]: g["gold_answer"] for g in U.read_jsonl(GOLD_RAD_FILE)}
    for c in CL.RQ1_OPEN_CELLS:
        if FB and c.startswith("CELL-M-"):
            continue
        for r in rad:
            if r["answer_kind"] != "open":
                continue
            a = ans[f"ANS|{c}|{r['qid']}|-"]
            pred = lead_text(a.get("text"))
            units.append(
                {
                    "key": f"J|{JUDGE}|J-OPEN|{a['key']}|main",
                    "task": "J-OPEN",
                    "answer_key": a["key"],
                    "qid": r["qid"],
                    "question": r["question"],
                    "gold": gold[r["qid"]],
                    "pred": pred,
                    "obs_status": "ok" if pred else "empty_prediction",
                }
            )
units = JG.shuffled(units, f"{JUDGE}|{'+'.join(TASKS)}")
if ax1_scope is not None:
    computed_plan = JG.build_ax1_plan(
        units, obs, JUDGE, ax1_scope_sha, U.sha256_file(PREP / probe_rel)
    )
    if computed_plan != ax1_plan:
        raise U.IntegrityError(
            (
                "REFERENCE_PANEL runtime units/observations "
                "differ from the locked judge plan; stop before "
                "model load"
            )
        )
if MODE == "PILOT":
    per = Counter()
    keep = []
    for u in units:
        if per[u["task"]] < PILOT_MAX_UNITS_PER_TASK:
            per[u["task"]] += 1
            keep.append(u)
    units = keep
keys = [u["key"] for u in units]
shard_keys = set(U.plan_shard(keys, NUM_SHARDS, SHARD_INDEX))
print(
    "units:",
    dict(
        Counter(
            (
                u["task"]
                + ("|probe" if u.get("is_probe_block") else "")
                + ("|rev" if u.get("variant") else "")
                for u in units
            )
        )
    ),
    "| skipped (partial observers):",
    dict(skipped_missing),
)

In [ ]:
t_load = time.time()
model = (
    B.StubVLM(JKEY, PRECISION)
    if BACKEND == "stub"
    else B.HFTextBackend(JKEY, PRECISION)
)
load_s = round(time.time() - t_load, 1)
session = U.ShardSession(
    "judge",
    RUN_ID,
    MODE,
    OUT_ROOT,
    SHARD_INDEX,
    NUM_SHARDS,
    PRIOR_OUTPUT_DIRS,
    time_limit_s=SHARD_TIME_LIMIT_S,
    max_items=MAX_ITEMS_THIS_SESSION,
)
if ax1_scope is not None:
    for source in (scope_path, plan_path):
        target = session.out_dir.parent / "artifacts" / source.name
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.is_file() and target.read_bytes() != source.read_bytes():
            raise U.IntegrityError(
                (
                    "REFERENCE_PANEL artifact differs in prior run; "
                    "use the approved fresh RUN_ID"
                )
            )
        if not target.is_file():
            target.write_bytes(source.read_bytes())
run_cfg_sha = RUN.write_run_config(
    session,
    {
        "notebook": "judge",
        "platform": dict(PLATFORM_CFG, lock=platform_report),
        "design": DESIGN,
        "rq2": RQ2,
        "mode": MODE,
        "judge": JUDGE,
        "model": model.identity,
        "tasks": TASKS,
        "observers": OBSERVERS,
        "partial_observers": PARTIAL_OBSERVERS,
        "max_new_tokens": JG.MAX_TOKENS,
        "precision": PRECISION,
        "generate_kwargs": B.GREEDY_KW,
        "diag_json_rule_strip_one_fence": JG.STRIP_ONE_JSON_FENCE,
        "order_seed": U.SEED,
        "load_s": load_s,
        "annotation_preflight": annotation_preflight,
        "ax1_scope_sha256": ax1_scope_sha,
        "ax1_plan_sha256": ax1_plan_sha,
        "env": U.environment_info(),
        "env_pins": env_report,
        "locked_inputs": lock_report,
    },
)
done, _ = session.done_keys()
todo = [u for u in units if u["key"] in shard_keys and u["key"] not in done]
print("model load s:", load_s, "| to do:", len(todo))
for u in todo:
    if session.should_stop():
        break
    rec = session.base("JUDGE-v1", u["key"])
    rec.update(
        {
            "judge": JUDGE,
            "task": u["task"],
            "variant": u.get("variant", "main"),
            "max_new_tokens": JG.MAX_TOKENS[u["task"]],
        }
    )
    t = u["task"]
    if t == "J-AX1":
        if ax1_scope is not None:
            rec.update(
                {
                    "ax1_scope_sha256": ax1_scope_sha,
                    "ax1_plan_sha256": ax1_plan_sha,
                    "ax1_image_id": u["nid"],
                }
            )
        rec["refs"] = {
            "obs_key": u["obs_key"],
            "is_probe_block": bool(u.get("is_probe_block")),
            "statements": [dict(st, id=i) for i, st in enumerate(u["stm"], 1)],
            "skipped_evasion": u["skipped_evasion"],
        }
        JG.assert_ax1_refs(rec["refs"])
        if u["obs_status"] != "ok" or not u["stm"]:
            rec.update(
                {
                    "status": "no_statements"
                    if u["obs_status"] == "ok"
                    else "obs_error_skipped",
                    "text": None,
                }
            )
            session.write(rec)
            continue
        prompt = JP.render_ax1(ann[u["nid"]]["text"], [st["text"] for st in u["stm"]])
    elif t == "J-AX2":
        rec["refs"] = {"obs_key": u["obs_key"]}
        if u["obs_status"] != "ok":
            rec.update({"status": "obs_error_skipped", "text": None})
            session.write(rec)
            continue
        routes = routing[u["nid"]]["routes"]
        prompt = JP.render_ax2(
            bank.grd_item_list(routes),
            JG.unsupported_for_routes(field_trace, routes),
            u["text"],
        )
    elif t == "J-AX3":
        rec["refs"] = {"obs_key": u["obs_key"], "qid": u["qid"]}
        if u["obs_status"] != "ok":
            rec.update({"status": "obs_error_skipped", "text": None})
            session.write(rec)
            continue
        prompt = JP.render_ax3(u["question"], u["text"])
    else:
        rec["refs"] = {
            "answer_key": u["answer_key"],
            "qid": u["qid"],
            "prediction": u["pred"],
        }
        if not u["pred"]:
            rec.update({"status": "empty_prediction", "text": None})
            session.write(rec)
            continue
        prompt = JP.render_open(u["question"], u["gold"], u["pred"])
    P.check3_no_original_names(prompt)
    if any(
        (
            x in prompt
            for x in [
                "OBS-L",
                "OBS-Q",
                "OBS-M",
                "OBS-G",
                "ARM-GRD",
                "ARM-ANA",
                "ARM-GEN",
            ]
        )
    ) or P.ORIGINAL_NAME_RE.search(prompt):
        raise P.LeakError("judge prompt not blind")
    ax1_ids = set(range(1, len(u["stm"]) + 1)) if t == "J-AX1" else None
    t0 = time.time()
    g = model.generate_text(prompt, JG.MAX_TOKENS[t])
    rec.update(
        {
            "status": "ok",
            "prompt_sha256": U.sha256_text(prompt),
            "text": g["text"],
            "truncated": g["truncated"],
            "diag_json_valid_strict": JG.json_valid_strict(t, g["text"], ax1_ids),
            "diag_json_valid": JG.json_valid(t, g["text"], ids=ax1_ids),
            "timing_s": round(time.time() - t0, 3),
        }
    )
    session.write(rec)

In [ ]:
recs = list(
    RUN.load_records([OUT_ROOT / RUN_ID] + list(PRIOR_OUTPUT_DIRS), "JUDGE-v1").values()
)
for r in recs:
    if r["task"] == "J-AX1":
        JG.assert_ax1_refs(r["refs"])
okr = [r for r in recs if r["status"] == "ok"]
tech = {"judge": JUDGE, "load_s": load_s, "by_task": {}}
for t in sorted({r["task"] for r in okr}):
    rr = [r for r in okr if r["task"] == t]
    ts = sorted((r["timing_s"] for r in rr))
    tech["by_task"][t] = {
        "n": len(rr),
        "diag_json_valid_rate": sum((r["diag_json_valid"] for r in rr)) / len(rr),
        "diag_json_valid_strict_rate": sum((r["diag_json_valid_strict"] for r in rr))
        / len(rr),
        "median_s": ts[len(ts) // 2],
        "truncated_rate": sum((bool(r["truncated"]) for r in rr)) / len(rr),
    }
rep = session.integrity(
    sorted(shard_keys),
    {
        "checks": {
            "check3": "pass",
            "blind": "pass",
            "check7": lock_report["verified"],
            "schema_ax1": "pass",
        },
        "status_counts": dict(Counter((r["status"] for r in recs))),
        "technical": tech,
        "run_config_sha256": run_cfg_sha,
    },
)
if MODE == "PILOT":
    U.write_json(session.out_dir / f"judge_pilot_report_{JUDGE}.json", tech)
print(
    {
        k: rep[k]
        for k in (
            "stopped_reason",
            "done_this_session",
            "remaining_in_shard",
            "duplicate_keys",
        )
    },
    rep["status_counts"],
)
print("technical:", tech)